# Домашнее задание: LLM-приложение с мониторингом Langfuse

В этом ноутбуке мы реализуем:
1. Простого чат-бота на базе **Google Gemini** (через AI Studio).
2. Интеграцию с **Langfuse** для трейсинга запросов.
3. Систему оценки (Evaluation) с использованием датасетов и LLM-as-a-judge.


In [30]:
# Установка зависимостей
%pip install --upgrade google-generativeai langfuse python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [31]:
import os
import time
from dotenv import load_dotenv
from langfuse import Langfuse
from langfuse.decorators import observe, langfuse_context
import google.generativeai as genai

# Загрузка переменных окружения
load_dotenv()

# Проверка ключей
if not os.getenv("GOOGLE_API_KEY"):
    print("WARNING: GOOGLE_API_KEY не найден!")
else:
    genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
    print("Google API Key загружен.")

# Инициализация Langfuse
langfuse = Langfuse()

# Проверка версии
import langfuse as lf_pkg
print(f"Langfuse SDK version: {lf_pkg.version.__version__}")

Google API Key загружен.
Langfuse SDK version: 3.12.0


## Часть 1: Реализация Чат-бота

Мы создадим класс `ChatApp`, который хранит историю диалога.
Мы используем **Google GenAI SDK** и вручную логируем параметры генерации (модель, токены) через `langfuse_context`.

In [ ]:
SYSTEM_PROMPT = "Ты полезный и вежливый AI-ассистент. Отвечай кратко и по делу."

class ChatApp:
    def __init__(self):
        self.history = [] 
        self.model = genai.GenerativeModel('gemini-flash-latest')
        self.chat_session = self.model.start_chat(history=[])
        
        # Отправляем системный промпт с использованием внутренней функции отправки с повтором
        self._send_message_safe(SYSTEM_PROMPT)

    def _send_message_safe(self, message, retries=5, initial_delay=10):
        """Отправка сообщения с автоматическим повтором при ошибке квоты (429)."""
        for attempt in range(retries):
            try:
                return self.chat_session.send_message(message)
            except Exception as e:
                # Если ошибка связана с квотой (429 или ResourceExhausted)
                if "429" in str(e) or "ResourceExhausted" in str(e):
                    if attempt < retries - 1:
                        sleep_time = initial_delay * (attempt + 1)
                        print(f"⚠️ Превышен лимит квоты Gemini. Ждем {sleep_time} сек...")
                        time.sleep(sleep_time)
                        continue
                raise e

    @observe(as_type="generation", name="llm_response")
    def get_llm_response(self, user_message):
        """
        Прямой вызов LLM (Gemini).
        """
        model_name = "gemini-flash-latest"
        
        # Вызов Gemini через безопасный метод
        try:
            response = self._send_message_safe(user_message)
            answer_text = response.text
        except Exception as e:
            return f"Error: Не удалось получить ответ от модели. {e}"
        
        input_tokens = 0
        output_tokens = 0
        
        try:
            # Считаем токены (это тоже API вызов, может упасть)
            input_tokens = self.model.count_tokens(user_message).total_tokens
            output_tokens = self.model.count_tokens(answer_text).total_tokens
        except:
            pass
        
        # Ручная регистрация параметров генерации в Langfuse
        langfuse_context.update_current_observation(
            model=model_name,
            usage={
                "input": input_tokens,
                "output": output_tokens,
                "total": input_tokens + output_tokens
            },
            input=user_message,
            output=answer_text
        )
        
        return answer_text

    @observe(name="chat_turn")
    def chat_turn(self, user_input):
        """
        Один шаг диалога.
        """
        # Трейс начнется автоматически благодаря декоратору @observe
        answer = self.get_llm_response(user_input)
        
        # Обогащаем трейс метаданными
        langfuse_context.update_current_trace(
            name=f"Chat: {user_input[:20]}...",
            user_id="gemini_user_1",
            tags=["gemini", "dev"]
        )
        
        return answer

# Пример использования
app = ChatApp()
print(f"User: Привет, как дела?")
response = app.chat_turn("Привет, как дела?")
print(f"Bot: {response}")

print(f"\nUser: Расскажи про Langfuse в двух словах.")
response = app.chat_turn("Расскажи про Langfuse в двух словах.")
print(f"Bot: {response}")

⚠️ Превышен лимит квоты Gemini. Ждем 10 сек...
⚠️ Превышен лимит квоты Gemini. Ждем 20 сек...


## Часть 2: Datasets и Experiments (Evaluation)

Теперь мы создадим **Dataset** (набор тестовых примеров) и запустим **Experiment**, чтобы оценить качество работы нашего бота.
Мы будем использовать **LLM-as-a-judge** (Gemini оценивает Gemini).

In [ ]:
# 1. Создание датасета
dataset_name = "Chatbot-Greeting-Benchmark-Notebook-v2"

# Получаем или создаем датасет
try:
    dataset = langfuse.get_dataset(dataset_name)
    print(f"Датасет '{dataset_name}' уже существует.")
except:
    print(f"Создаем датасет '{dataset_name}'...")
    langfuse.create_dataset(
        name=dataset_name,
        description="Тестовый набор приветствий"
    )
    # Добавляем примеры
    langfuse.create_dataset_item(
        dataset_name=dataset_name,
        input={"message": "Привет, кто ты?"},
        expected_output="Я AI-ассистент."
    )
    langfuse.create_dataset_item(
        dataset_name=dataset_name,
        input={"message": "Как дела?"},
        expected_output="Я в порядке, как искусственный интеллект."
    )
    langfuse.create_dataset_item(
        dataset_name=dataset_name,
        input={"message": "Ты умеешь программировать?"},
        expected_output="Да, я могу помочь с кодом."
    )
    print("Датасет создан!")
    # Перезагружаем датасет, чтобы получить items
    dataset = langfuse.get_dataset(dataset_name)

Датасет 'Chatbot-Greeting-Benchmark-Notebook-v2' уже существует.


In [ ]:
# Helper for retry
def generate_content_with_retry(model, prompt, retries=5, initial_delay=10):
    for attempt in range(retries):
        try:
            return model.generate_content(prompt)
        except Exception as e:
            if "429" in str(e) or "ResourceExhausted" in str(e):
                if attempt < retries - 1:
                    sleep_time = initial_delay * (attempt + 1)
                    print(f"⚠️ Quota exceeded (Eval). Ждем {sleep_time} сек...")
                    time.sleep(sleep_time)
                    continue
            raise e

# 2. Функция Эвалюатор (LLM Judge с помощью Gemini)
def evaluate_response(input_text, output_text, expected_output):
    """
    Оценивает ответ чатбота по шкале от 0 до 1 (вежливость и релевантность).
    """
    evaluation_prompt = f"""
    Please act as an impartial judge and evaluate the quality of the response provided by an AI assistant.
    
    Input: {input_text}
    Expected intent: {expected_output}
    Actual Response: {output_text}
    
    Is the actual response polite and relevant to the input? 
    Return ONLY a number: 
    1.0 = Perfect
    0.5 = Okay
    0.0 = Bad
    """
    
    model = genai.GenerativeModel('gemini-flash-latest')
    
    try:
        response = generate_content_with_retry(model, evaluation_prompt)
        content = response.text.strip()
        return float(content)
    except:
        return 0.5

# 3. Функция, которую мы тестируем (для эксперимента)
@observe(name="generation_for_eval")
def simple_chat_func(message):
    model = genai.GenerativeModel('gemini-flash-latest')
    
    # Используем наш хелпер с ретраем
    try:
        response = generate_content_with_retry(model, message)
        text = response.text
    except Exception as e:
        text = f"Error: {e}"
        
    # Логируем
    langfuse_context.update_current_observation(
        model="gemini-flash-latest",
        input=message,
        output=text
    )
    return text

In [ ]:
# 4. Запуск Эксперимента
print("🚀 Запускаем эксперимент (Gemini)...")

# Инициализируем модель вне цикла
eval_model = genai.GenerativeModel('gemini-flash-latest')

for item in dataset.items:
    input_msg = item.input["message"]
    
    # Use start_generation directly as the main observation
    generation = langfuse.start_generation(
        name="chat-generation",
        model="gemini-flash-latest",
        input=input_msg,
        metadata={"experiment": "Gemini-Notebook-Exp"}
    )
    
    try:
        # Вызов Gemini с ретраем (используем функцию из предыдущей ячейки)
        response = generate_content_with_retry(eval_model, input_msg)
        answer = response.text
        
        # Завершаем генерацию
        generation.update(output=answer)
        generation.end()
        
        # Link to dataset (requires compatible SDK version or method)
        # item.link(
        #     trace_or_observation=generation,
        #     run_name="Gemini-Notebook-Exp",
        # )
        
        # Оценка
        score_value = evaluate_response(input_msg, answer, item.expected_output)
        
        generation.score(
            name="politeness",
            value=score_value,
            comment="Gemini Judge Auto-eval"
        )
        
        print(f"Msg: {input_msg} | Score: {score_value}")
        
    except Exception as e:
        print(f"Error processing item: {e}")
        # Only end if not already ended or check status - simplified:
        try:
            generation.update(level="ERROR", status_message=str(e))
            generation.end()
        except:
            pass

print("\n✅ Эксперимент завершен!")

🚀 Запускаем эксперимент (Gemini)...


/tmp/ipykernel_10304/4000332661.py:11: DeprecationWarning: start_generation is deprecated and will be removed in a future version. Use start_observation(as_type='generation') instead.
  generation = langfuse.start_generation(


Msg: Ты умеешь программировать? | Score: 1.0


Calling end() on an ended span.


Error processing item: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash
Please retry in 43.493902003s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 5
}
, retry_delay {
  seconds: 43
}
]
Error processing item: 429 You exceeded your current quota, please check your plan an